# Revisar Cobranzas con pandas

Notebook simple para:
- leer el Excel con `pandas`
- correr la simulación local con `simulate_drive_upload_local.js`
- traer todos los leads de `Kommo`
- comparar `Excel vs Kommo`


In [11]:
import json
import subprocess
import tempfile
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)


In [12]:
excel_path = Path(r"C:\Users\luis_\Downloads\cxc 16-4-26 kommo act.xlsx")
run_date_iso = "2026-04-18"
env_path = Path(".env")
script_path = Path("simulate_drive_upload_local.js").resolve()

node_candidates = [
    Path(r"C:\Program Files\nodejs\node.exe"),
    Path("/mnt/c/Program Files/nodejs/node.exe"),
]
node_bin = next((str(path) for path in node_candidates if path.exists()), "node")

kommo_pipeline_id = 13403731
kommo_tag = "cobranza_n8n_excel"

excel_path


WindowsPath('C:/Users/luis_/Downloads/cxc 16-4-26 kommo act.xlsx')

In [13]:
env = {}
for raw_line in env_path.read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    env[key.strip()] = value.strip().strip('"').strip("'")

kommo_base = env["KOMMO_BASE"].rstrip("/")
kommo_token = env["KOMMO_TOKEN"]

pd.DataFrame([
    {"key": "excel_path", "value": str(excel_path)},
    {"key": "run_date_iso", "value": run_date_iso},
    {"key": "script_path", "value": str(script_path)},
    {"key": "kommo_base", "value": kommo_base},
])


,key,value
0,excel_path,C:\Users\luis_\Downloads\cxc 16-4-26 kommo act...
1,run_date_iso,2026-04-18
2,script_path,C:\Users\luis_\OneDrive\Desktop\KommoAgroriego...
3,kommo_base,https://agroriegoscorp.kommo.com


In [14]:
excel_df = pd.read_excel(excel_path).fillna("")
excel_df.columns = [str(column).strip() for column in excel_df.columns]

display(excel_df.head(10))
display(pd.DataFrame([{"metric": "excel_rows", "value": len(excel_df)}]))


,VENDEDOR,COD VEN,COD CLI,RAZON SOCIAL,TELEFONO1,DOCUMENTO,SUCURSAL,NOMBRE SUCURSAL,TIPO DOC,FECHA,FECHA RECEPCION,PLAZO,FECHA VENC,DIAS VENCIDOS,NETO,SALDO,MONTO USD,SALDO DOC,PAGO
0,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,708,ADRIAN EDUARDO ARELLANO FARIAS/CONSTRUCTORA E ...,584140804385,129,1,Centro de Costos,ANTV,2026-03-20,00-00-0000,30,2026-04-19,-3,-1069.02,-112.02,-1069.02,-112.02,957.00
1,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,790,AGRO ANIMAL LOS MONTOYA C.A,584124012262,3065,1,Centro de Costos,NEN,2026-03-11,2026-03-16 00:00:00,30,2026-04-15,1,853.73,2.81,853.73,2.81,850.92
2,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,759,AGROPECUARIA Y MATERIALES DE CONSTRUCCION LOS...,584167789743,2929,1,Centro de Costos,NEN,2026-02-25,2026-02-27 00:00:00,30,2026-03-29,18,496.40,1.40,496.40,1.40,495.00
3,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,925,AMAYA SUAREZ INVERSIONES CRISYO C.A,584126762007,2930,1,Centro de Costos,NEN,2026-02-25,2026-02-27 00:00:00,30,2026-03-29,18,1756.82,335.82,1756.82,335.82,1421.00
4,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,731,ARELLANO PARRA CARMEN JAVIER/FERRETODO LA FORT...,584167053035,3068,1,Centro de Costos,NEN,2026-03-11,2026-03-16 00:00:00,30,2026-04-15,1,84.50,84.50,84.50,84.50,0.00
5,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,731,ARELLANO PARRA CARMEN JAVIER/FERRETODO LA FORT...,584167053035,2657,1,Centro de Costos,NEN,2026-02-02,2026-02-09 00:00:00,30,2026-03-11,36,215.50,30.50,215.50,30.50,185.00
6,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,835,BARBARA RITA PERALTA GARCIA / FERRE AGRO DON M...,584245932937,3074,1,Centro de Costos,NEN,2026-03-12,2026-03-16 00:00:00,30,2026-04-15,1,945.15,945.15,945.15,945.15,0.00
7,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,835,BARBARA RITA PERALTA GARCIA / FERRE AGRO DON M...,584245932937,3095,1,Centro de Costos,NEN,2026-03-12,2026-03-16 00:00:00,30,2026-04-15,1,1012.81,1012.81,1012.81,1012.81,0.00
8,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,586,BERNAVE SALAS MARQUINA / TRUPERMAX,584245772813,90000001261,1,Centro de Costos,NEND,2025-04-03,00-00-0000,30,2025-05-03,348,369.46,248.46,369.46,248.46,121.00
9,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,930,CARLOS ANDRES CARDENAS RIOS FERRE AGRO CARDENAS,584247432779,257,1,Centro de Costos,DEVNE,2026-03-16,00-00-0000,30,2026-04-15,1,-99.34,-99.34,-99.34,-99.34,0.00


,metric,value
0,excel_rows,350


In [15]:
tmp_dir = Path(tempfile.mkdtemp(prefix="sim_cobranza_"))
json_out = tmp_dir / "sim_result.json"
csv_out = tmp_dir / "sim_result.csv"

script_arg = str(script_path)
excel_arg = str(excel_path)
json_arg = str(json_out)
csv_arg = str(csv_out)

if str(node_bin).lower().endswith('.exe'):
    if script_arg.startswith('/mnt/'):
        drive = script_arg.split('/')[2].upper()
        rest = '\\'.join(script_arg.split('/')[3:])
        script_arg = f"{drive}:\\{rest}"
    if excel_arg.startswith('/mnt/'):
        drive = excel_arg.split('/')[2].upper()
        rest = '\\'.join(excel_arg.split('/')[3:])
        excel_arg = f"{drive}:\\{rest}"
    if json_arg.startswith('/mnt/'):
        drive = json_arg.split('/')[2].upper()
        rest = '\\'.join(json_arg.split('/')[3:])
        json_arg = f"{drive}:\\{rest}"
    if csv_arg.startswith('/mnt/'):
        drive = csv_arg.split('/')[2].upper()
        rest = '\\'.join(csv_arg.split('/')[3:])
        csv_arg = f"{drive}:\\{rest}"

cmd = [
    node_bin,
    script_arg,
    "--excel", excel_arg,
    "--date", run_date_iso,
    "--json-out", json_arg,
    "--csv-out", csv_arg,
]

completed = subprocess.run(cmd, capture_output=True, text=True)
if completed.returncode != 0:
    raise RuntimeError(completed.stdout + "\n\n" + completed.stderr)

sim_result = json.loads(json_out.read_text(encoding="utf-8"))
sim_result["counts"]


{'source_rows': 350,
 'clean_rows': 350,
 'upsert_created': 303,
 'upsert_updated': 0,
 'upsert_skipped': 47,
 'reminder_moves': 73}

In [16]:
counts_df = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in sim_result["counts"].items()
])

reminder_df = pd.DataFrame(sim_result.get("reminder_logs", []))
if not reminder_df.empty and "documento" in reminder_df.columns:
    reminder_df["documento"] = reminder_df["documento"].astype(str).str.strip()

skipped_df = pd.DataFrame(sim_result.get("skipped_upserts", []))
without_move_df = pd.DataFrame(sim_result.get("upserted_without_movement", []))

display(counts_df)
display(reminder_df)


,metric,value
0,source_rows,350
1,clean_rows,350
2,upsert_created,303
3,upsert_updated,0
4,upsert_skipped,47
5,reminder_moves,73


,lead_id,documento,razon_social,telefono,saldo_pendiente,fecha_venc_iso,current_status_id,current_status_name,days_to_due,reminder_type,stage_target,stage_target_name,reminder_field_id,reminder_field_name,would_move_stage
0,90000000210,2964,TORNIFERREAGRO KEMEL C.A,5841511,56.50,2026-04-04,103388967,leads_importados,-14,LATE_10D,104161951,atrasado_10d,0,,True
1,90000000042,1866,FOGONERA MARKET C.A,58424711,268.05,2025-10-31,103388967,leads_importados,-169,LATE_15D,104161955,atrasado_15d,0,,True
2,90000000205,2503,MILAGROS DEL CARMEN CARDENAS MONTOYA,58424711,321.54,2026-02-25,103388967,leads_importados,-52,LATE_15D,104161955,atrasado_15d,0,,True
3,90000000216,2609,JOSE DANIEL ASCANIO VALECILLOS,58424511,17.61,2026-02-27,103388967,leads_importados,-50,LATE_15D,104161955,atrasado_15d,0,,True
4,90000000213,2697,CLIENTE EVENTUAL BARINAS,58424811,114.94,2026-03-07,103388967,leads_importados,-42,LATE_15D,104161955,atrasado_15d,0,,True
5,90000000045,2734,INVERSIONES AGROFERRE EXPRESS C.A,58424611,1360.35,2026-03-14,103388967,leads_importados,-35,LATE_15D,104161955,atrasado_15d,0,,True
6,90000000199,2796,GHERSY RHENNYER SANCHEZ ROSALES / FERRERIEGO,58424711,70.00,2026-03-21,103388967,leads_importados,-28,LATE_15D,104161955,atrasado_15d,0,,True
7,90000000044,2831,INVERSIONES AGROFERRE EXPRESS C.A,58424611,569.45,2026-03-22,103388967,leads_importados,-27,LATE_15D,104161955,atrasado_15d,0,,True
8,90000000072,2833,TOUFIK JOSE SALAH ORTEGA,58424811,854.84,2026-03-22,103388967,leads_importados,-27,LATE_15D,104161955,atrasado_15d,0,,True
9,90000000192,2849,ANTONIO MANUEL DE OLIVEIRA E CIRNE / FERREAGRO...,58424811,321.55,2026-03-27,103388967,leads_importados,-22,LATE_15D,104161955,atrasado_15d,0,,True


LLAMADA A LA API DE KOMMO

In [17]:
headers = {
    "Authorization": f"Bearer {kommo_token}",
    "Accept": "application/json",
}

kommo_all_leads = []
for page in range(1, 101):
    query = urllib.parse.urlencode({
        "limit": 250,
        "page": page,
        "with": "custom_fields_values,tags",
    })
    url = kommo_base + "/api/v4/leads?" + query
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request, timeout=60) as response:
        body = response.read()
    data = json.loads(body.decode("utf-8")) if body else {}
    chunk = (((data or {}).get("_embedded") or {}).get("leads") or [])
    if not chunk:
        break
    kommo_all_leads.extend(chunk)
    if not ((((data or {}).get("_links") or {}).get("next") or {}).get("href")):
        break

pd.DataFrame([{"metric": "kommo_all_leads", "value": len(kommo_all_leads)}])


,metric,value
0,kommo_all_leads,303


In [18]:
field_ids = {
    "documento": 3272952,
    "telefono": 3281414,
    "fecha_venc_text": 3281430,
    "saldo_pendiente": 3272956,
    "pago_realizado": 3281416,
    "razon_social": 3281418,
    "fecha_venc_date": 3272954,
    "aviso_3d": 3282254,
    "aviso_2d": 3282256,
    "aviso_1d": 3282258,
}

stage_names = {
    103388967: "leads_importados",
    103388971: "recordatorio_enviado",
    103388975: "fecha_limite",
    103388979: "atrasado_5d",
    104161951: "atrasado_10d",
    104161955: "atrasado_15d",
    103429439: "pagado",
    103610175: "abono",
    103611443: "no_pagado",
    103429435: "revisar_pago",
}

kommo_rows = []
for lead in kommo_all_leads:
    fields = {}
    for item in lead.get("custom_fields_values") or []:
        values = item.get("values") or []
        value = ""
        if values:
            raw_value = values[0].get("value")
            value = "" if raw_value is None else str(raw_value)
        fields[int(item.get("field_id") or 0)] = value

    tags = [str(item.get("name") or "").strip() for item in ((lead.get("_embedded") or {}).get("tags") or [])]

    kommo_rows.append({
        "lead_id": int(lead.get("id") or 0),
        "name": str(lead.get("name") or ""),
        "pipeline_id": int(lead.get("pipeline_id") or 0),
        "status_id": int(lead.get("status_id") or 0),
        "status_name": stage_names.get(int(lead.get("status_id") or 0), str(lead.get("status_id") or 0)),
        "documento": str(fields.get(field_ids["documento"], "")).strip(),
        "telefono": str(fields.get(field_ids["telefono"], "")).strip(),
        "razon_social": str(fields.get(field_ids["razon_social"], "")).strip(),
        "fecha_venc_text": str(fields.get(field_ids["fecha_venc_text"], "")).strip(),
        "fecha_venc_date_raw": str(fields.get(field_ids["fecha_venc_date"], "")).strip(),
        "saldo_pendiente": str(fields.get(field_ids["saldo_pendiente"], "")).strip(),
        "pago_realizado": str(fields.get(field_ids["pago_realizado"], "")).strip(),
        "aviso_3d": str(fields.get(field_ids["aviso_3d"], "")).strip(),
        "aviso_2d": str(fields.get(field_ids["aviso_2d"], "")).strip(),
        "aviso_1d": str(fields.get(field_ids["aviso_1d"], "")).strip(),
        "tags": ", ".join(tags),
        "updated_at": int(lead.get("updated_at") or 0),
        "created_at": int(lead.get("created_at") or 0),
    })

kommo_df = pd.DataFrame(kommo_rows)
kommo_df["saldo_pendiente"] = pd.to_numeric(kommo_df["saldo_pendiente"], errors="coerce").fillna(0.0)
kommo_df["pago_realizado"] = pd.to_numeric(kommo_df["pago_realizado"], errors="coerce").fillna(0.0)
kommo_df["fecha_venc_iso"] = ""

mask_ts = kommo_df["fecha_venc_date_raw"].astype(str).str.fullmatch(r"\d+", na=False)
kommo_df.loc[mask_ts, "fecha_venc_iso"] = pd.to_datetime(
    kommo_df.loc[mask_ts, "fecha_venc_date_raw"].astype("int64"),
    unit="s",
    utc=True,
).dt.strftime("%Y-%m-%d")

mask_text = kommo_df["fecha_venc_iso"].eq("") & kommo_df["fecha_venc_text"].ne("")
kommo_df.loc[mask_text, "fecha_venc_iso"] = pd.to_datetime(
    kommo_df.loc[mask_text, "fecha_venc_text"],
    dayfirst=True,
    errors="coerce",
).dt.strftime("%Y-%m-%d").fillna("")

kommo_df["has_cobranza_tag"] = kommo_df["tags"].str.contains(kommo_tag, na=False)
kommo_df["is_pipeline_cobranza"] = kommo_df["pipeline_id"].eq(kommo_pipeline_id)

display(kommo_df.head(10))


,lead_id,name,pipeline_id,status_id,status_name,documento,telefono,razon_social,fecha_venc_text,fecha_venc_date_raw,saldo_pendiente,pago_realizado,aviso_3d,aviso_2d,aviso_1d,tags,updated_at,created_at,fecha_venc_iso,has_cobranza_tag,is_pipeline_cobranza
0,61213892,Factura 3065 - AGRO ANIMAL LOS MONTOYA C.A,13403731,103388967,leads_importados,3065,584124012262,AGRO ANIMAL LOS MONTOYA C.A,15/04/2026,1776232800,0.00,850.92,,,,"cobranza_n8n_excel, Ruta 4",1776541027,1776541027,2026-04-15,True,True
1,61213894,Factura 2929 - AGROPECUARIA Y MATERIALES DE C...,13403731,103388967,leads_importados,2929,584167789743,AGROPECUARIA Y MATERIALES DE CONSTRUCCION LOS...,29/03/2026,1774764000,0.00,495.00,,,,"cobranza_n8n_excel, Ruta 4",1776541031,1776541031,2026-03-29,True,True
2,61213898,Factura 2930 - AMAYA SUAREZ INVERSIONES CRISYO...,13403731,103388967,leads_importados,2930,584126762007,AMAYA SUAREZ INVERSIONES CRISYO C.A,29/03/2026,1774764000,0.00,1421.00,,,,"cobranza_n8n_excel, Ruta 4",1776541034,1776541034,2026-03-29,True,True
3,61213902,Factura 2657 - ARELLANO PARRA CARMEN JAVIER/FE...,13403731,103388967,leads_importados,2657,584167053035,ARELLANO PARRA CARMEN JAVIER/FERRETODO LA FORT...,11/03/2026,1773208800,0.00,185.00,,,,"cobranza_n8n_excel, Ruta 4",1776541041,1776541041,2026-03-11,True,True
4,61213928,Factura 3077 - CARLOS ANDRES CARDENAS RIOS FER...,13403731,103388967,leads_importados,3077,584247432779,CARLOS ANDRES CARDENAS RIOS FERRE AGRO CARDENAS,15/04/2026,1776232800,0.00,2475.00,,,,"cobranza_n8n_excel, Ruta 4",1776541056,1776541056,2026-04-15,True,True
5,61213934,Factura 15 - CARLOS ARTURO SILVA MOGOLLON/AGRO...,13403731,103388967,leads_importados,15,584264734697,CARLOS ARTURO SILVA MOGOLLON/AGROFER SILVA 2015,20/07/2025,1752991200,0.00,554.29,,,,"cobranza_n8n_excel, Ruta 4",1776541059,1776541059,2025-07-20,True,True
6,61213938,Factura 90000001396 - CARLOS ARTURO SILVA MOGO...,13403731,103388967,leads_importados,90000001396,584264734697,CARLOS ARTURO SILVA MOGOLLON/AGROFER SILVA 2015,13/06/2025,1749794400,0.00,515.20,,,,"cobranza_n8n_excel, Ruta 4",1776541062,1776541062,2025-06-13,True,True
7,61213948,Factura 3330 - CARLOS EDUADO ARAQUE ZAPATA / F...,13403731,103388967,leads_importados,3330,584161787556,CARLOS EDUADO ARAQUE ZAPATA / FRIOVENCA ARAQUE,13/05/2026,1778652000,702.41,0.00,,,,"cobranza_n8n_excel, Ruta 4",1776541066,1776541066,2026-05-13,True,True
8,61213950,Factura 3229 - CARLOS FAGED SALAH ORTEGA (INVE...,13403731,103388967,leads_importados,3229,584165025187,CARLOS FAGED SALAH ORTEGA (INVERSIONES AGRO HO...,13/05/2026,1778652000,431.41,79.42,,,,"cobranza_n8n_excel, Ruta 4",1776541069,1776541069,2026-05-13,True,True
9,61213954,"Factura 3235 - CARMEN YURITZA PLANA ""FERRELECT...",13403731,103388967,leads_importados,3235,573022227530,"CARMEN YURITZA PLANA ""FERRELECTRICO SOFI PLANA""",13/05/2026,1778652000,927.50,0.00,,,,"cobranza_n8n_excel, Ruta 4",1776541074,1776541074,2026-05-13,True,True


In [19]:
kommo_cobranza_df = kommo_df.loc[
    kommo_df["has_cobranza_tag"] & kommo_df["is_pipeline_cobranza"]
].copy()

kommo_cobranza_df = kommo_cobranza_df.sort_values(["documento", "updated_at", "lead_id"])
kommo_cobranza_df = kommo_cobranza_df.drop_duplicates("documento", keep="last")

display(pd.DataFrame([
    {"metric": "kommo_all_leads", "value": len(kommo_df)},
    {"metric": "kommo_cobranza_docs", "value": len(kommo_cobranza_df)},
]))

display(kommo_cobranza_df)


,metric,value
0,kommo_all_leads,303
1,kommo_cobranza_docs,303


,lead_id,name,pipeline_id,status_id,status_name,documento,telefono,razon_social,fecha_venc_text,fecha_venc_date_raw,saldo_pendiente,pago_realizado,aviso_3d,aviso_2d,aviso_1d,tags,updated_at,created_at,fecha_venc_iso,has_cobranza_tag,is_pipeline_cobranza
5,61213934,Factura 15 - CARLOS ARTURO SILVA MOGOLLON/AGRO...,13403731,103388967,leads_importados,15,584264734697,CARLOS ARTURO SILVA MOGOLLON/AGROFER SILVA 2015,20/07/2025,1752991200,0.00,554.29,,,,"cobranza_n8n_excel, Ruta 4",1776541059,1776541059,2025-07-20,True,True
203,61215116,Factura 1558 - ORLANDO ALMEIDA MARTINEZ/CONEXI...,13403731,103388967,leads_importados,1558,584247309038,ORLANDO ALMEIDA MARTINEZ/CONEXIONES Y TUBERIAS...,27/09/2025,1758952800,0.00,5640.15,,,,"cobranza_n8n_excel, Ruta 8",1776541860,1776541860,2025-09-27,True,True
206,61215130,Factura 1647 - FERTILIZANTES LAS CANALES C.A,13403731,103388967,leads_importados,1647,584247114397,FERTILIZANTES LAS CANALES C.A,21/09/2025,1758434400,0.00,237.00,,,,"cobranza_n8n_excel, Ruta 10",1776541875,1776541875,2025-09-21,True,True
44,61214122,Factura 1866 - FOGONERA MARKET C.A,13403731,104161955,atrasado_15d,1866,584247283144,FOGONERA MARKET C.A,31/10/2025,1761890400,268.05,0.00,,,,"cobranza_n8n_excel, Ruta 4",1776541216,1776541190,2025-10-31,True,True
70,61214258,Factura 19 - RICHARD ANTONIO PAEZ VIVAS / FERR...,13403731,103388967,leads_importados,19,584261376418,RICHARD ANTONIO PAEZ VIVAS / FERREELECTRICOS L...,27/07/2025,1753596000,0.00,6871.68,,,,"cobranza_n8n_excel, Ruta 4",1776541303,1776541303,2025-07-27,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,61215612,Factura 90000000672 - ALAIN JESUS MONTAÑEZ CON...,13403731,104161955,atrasado_15d,90000000672,584161938673,ALAIN JESUS MONTAÑEZ CONTRERAS,06/11/2024,1730872800,177.80,0.00,,,,"cobranza_n8n_excel, Ruta 14",1776542268,1776542252,2024-11-06,True,True
302,61215634,Factura 90000000696 - SUPERMERCADO LOS RUICES ...,13403731,104161955,atrasado_15d,90000000696,584264720510,"SUPERMERCADO LOS RUICES EL MORALITO, C.A.",15/11/2024,1731650400,191.32,0.00,,,,"cobranza_n8n_excel, Ruta 14",1776542269,1776542265,2024-11-15,True,True
299,61215626,Factura 90000000980 - JACINTO ANTONIO LARA VER...,13403731,103388967,leads_importados,90000000980,584145000415,JACINTO ANTONIO LARA VERGARA / VARIEDADES JACI...,08/03/2025,1741413600,0.00,180.00,,,,"cobranza_n8n_excel, Ruta 14",1776542261,1776542261,2025-03-08,True,True
13,61213920,Factura 90000001261 - BERNAVE SALAS MARQUINA /...,13403731,104161955,atrasado_15d,90000001261,584245772813,BERNAVE SALAS MARQUINA / TRUPERMAX,03/05/2025,1746252000,127.46,121.00,,,,"cobranza_n8n_excel, Ruta 4",1776541090,1776541052,2025-05-03,True,True


In [20]:
doc_col = next((col for col in ["DOCUMENTO", "Documento"] if col in excel_df.columns), None)
saldo_col = next((col for col in ["SALDO", "Saldo", "saldo"] if col in excel_df.columns), None)

expected_df = excel_df.copy()
expected_df = expected_df.loc[expected_df[doc_col].astype(str).str.strip().ne("")].copy()
expected_df["documento"] = pd.to_numeric(expected_df[doc_col], errors="coerce").astype("Int64").astype(str)
expected_df = expected_df.loc[expected_df["documento"].ne("<NA>")].copy()
expected_df["saldo_excel"] = pd.to_numeric(expected_df[saldo_col], errors="coerce").fillna(0)
expected_df = expected_df.loc[expected_df["saldo_excel"] > 0].copy()
expected_df = expected_df.drop_duplicates("documento", keep="last")

expected_df["expected_status_id"] = 103388967
expected_df["expected_status_name"] = "leads_importados"
expected_df["expected_reminder_type"] = ""
expected_df["expected_days_to_due"] = pd.NA

if not reminder_df.empty:
    expected_df = expected_df.merge(
        reminder_df[["documento", "stage_target", "stage_target_name", "reminder_type", "days_to_due"]].drop_duplicates("documento"),
        on="documento",
        how="left",
    )
    expected_df["expected_status_id"] = expected_df["stage_target"].fillna(expected_df["expected_status_id"]).astype(int)
    expected_df["expected_status_name"] = expected_df["stage_target_name"].fillna(expected_df["expected_status_name"])
    expected_df["expected_reminder_type"] = expected_df["reminder_type"].fillna("")
    expected_df["expected_days_to_due"] = expected_df["days_to_due"]

comparison_df = expected_df.merge(
    kommo_cobranza_df[[
        "documento",
        "lead_id",
        "razon_social",
        "telefono",
        "status_id",
        "status_name",
        "fecha_venc_iso",
        "saldo_pendiente",
        "pago_realizado",
        "aviso_3d",
        "aviso_2d",
        "aviso_1d",
    ]],
    on="documento",
    how="outer",
    indicator=True,
)

comparison_df["expected_status_name"] = comparison_df["expected_status_name"].fillna("unexpected_in_kommo")
comparison_df["actual_status_name"] = comparison_df["status_name"].fillna("missing_in_kommo")
comparison_df["ok"] = "NO_OK"

mask_ok = (
    comparison_df["_merge"].eq("both")
    & comparison_df["expected_status_id"].fillna(-1).astype(int).eq(comparison_df["status_id"].fillna(-2).astype(int))
)
comparison_df.loc[mask_ok, "ok"] = "OK"

comparison_df = comparison_df.sort_values(["ok", "expected_status_name", "documento"], ascending=[True, True, True])

summary_compare_df = pd.DataFrame([
    {"metric": "expected_docs", "value": expected_df["documento"].nunique()},
    {"metric": "actual_cobranza_docs", "value": kommo_cobranza_df["documento"].nunique()},
    {"metric": "ok", "value": int(comparison_df["ok"].eq("OK").sum())},
    {"metric": "no_ok", "value": int(comparison_df["ok"].eq("NO_OK").sum())},
    {"metric": "missing_in_kommo", "value": int(comparison_df["_merge"].eq("left_only").sum())},
    {"metric": "unexpected_in_kommo", "value": int(comparison_df["_merge"].eq("right_only").sum())},
])

display(summary_compare_df)
display(comparison_df[[
    "ok",
    "documento",
    "lead_id",
    "razon_social",
    "expected_status_name",
    "actual_status_name",
    "expected_reminder_type",
    "expected_days_to_due",
    "fecha_venc_iso",
    "saldo_pendiente",
    "pago_realizado",
    "aviso_3d",
    "aviso_2d",
    "aviso_1d",
]])


,metric,value
0,expected_docs,303
1,actual_cobranza_docs,303
2,ok,303
3,no_ok,0
4,missing_in_kommo,0
5,unexpected_in_kommo,0


,ok,documento,lead_id,razon_social,expected_status_name,actual_status_name,expected_reminder_type,expected_days_to_due,fecha_venc_iso,saldo_pendiente,pago_realizado,aviso_3d,aviso_2d,aviso_1d
45,OK,2964,61215122,TORNIFERREAGRO KEMEL C.A,atrasado_10d,atrasado_10d,LATE_10D,-14.0,2026-04-04,56.50,0.0,,,
3,OK,1866,61214122,FOGONERA MARKET C.A,atrasado_15d,atrasado_15d,LATE_15D,-169.0,2025-10-31,268.05,0.0,,,
9,OK,2503,61215108,MILAGROS DEL CARMEN CARDENAS MONTOYA,atrasado_15d,atrasado_15d,LATE_15D,-52.0,2026-02-25,321.54,0.0,,,
16,OK,2609,61215158,JOSE DANIEL ASCANIO VALECILLOS,atrasado_15d,atrasado_15d,LATE_15D,-50.0,2026-02-27,17.61,0.0,,,
20,OK,2697,61215142,CLIENTE EVENTUAL BARINAS,atrasado_15d,atrasado_15d,LATE_15D,-42.0,2026-03-07,114.94,0.0,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,OK,3133,61215258,FERRETERIA Y MATERIALES CONSTRUCCIONES EXPRESS...,recordatorio_enviado,recordatorio_enviado,5D,5.0,2026-04-23,105.66,0.0,2026-04-18T19:52:54+00:00,,
117,OK,3134,61215504,ESPERANZA AL KATAR DARWEICH,recordatorio_enviado,recordatorio_enviado,5D,5.0,2026-04-23,762.40,0.0,2026-04-18T19:57:00+00:00,,
119,OK,3140,61215200,"EL RINCON DEL PRODUCTOR 2018, CA",recordatorio_enviado,recordatorio_enviado,5D,5.0,2026-04-23,661.42,0.0,2026-04-18T19:52:50+00:00,,
121,OK,3142,61215424,ELECTRONICA COMERCIAL PODER NEGRO C.A,recordatorio_enviado,recordatorio_enviado,5D,5.0,2026-04-23,257.97,0.0,2026-04-18T19:55:43+00:00,,


In [21]:
mismatches_df = comparison_df.loc[comparison_df["ok"].eq("NO_OK")].copy()
display(mismatches_df)

display(skipped_df)
display(without_move_df)


,VENDEDOR,COD VEN,COD CLI,RAZON SOCIAL,TELEFONO1,DOCUMENTO,SUCURSAL,NOMBRE SUCURSAL,TIPO DOC,FECHA,FECHA RECEPCION,PLAZO,FECHA VENC,DIAS VENCIDOS,NETO,SALDO,MONTO USD,SALDO DOC,PAGO,documento,saldo_excel,expected_status_id,expected_status_name,expected_reminder_type,expected_days_to_due,stage_target,stage_target_name,reminder_type,days_to_due,lead_id,razon_social,telefono,status_id,status_name,fecha_venc_iso,saldo_pendiente,pago_realizado,aviso_3d,aviso_2d,aviso_1d,_merge,actual_status_name,ok


,VENDEDOR,COD_VEN,DOCUMENTO,RAZON_SOCIAL,TELEFONO,SALDO_DOC,FECHA_VENC,FECHA_VENC_ISO,DIAS_VENCIDOS,MONTO_USD,PAGO,TITULO_TRATO,upsert_status,upsert_reason,upsert_saldo_doc
0,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,129,ADRIAN EDUARDO ARELLANO FARIAS/CONSTRUCTORA E ...,58414111,-112.02,19/04/2026,2026-04-19,-3,-1069.02,957.00,Factura 129 - ADRIAN EDUARDO ARELLANO FARIAS/C...,skipped,saldo_no_positivo,-112.02
1,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,257,CARLOS ANDRES CARDENAS RIOS FERRE AGRO CARDENAS,58424711,-99.34,15/04/2026,2026-04-15,1,-99.34,0.00,Factura 257 - CARLOS ANDRES CARDENAS RIOS FERR...,skipped,saldo_no_positivo,-99.34
2,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,69,"CARMEN YURITZA PLANA ""FERRELECTRICO SOFI PLANA""",57302211,-17.84,16/01/2026,2026-01-16,90,-17.84,0.00,"Factura 69 - CARMEN YURITZA PLANA ""FERRELECTRI...",skipped,saldo_no_positivo,-17.84
3,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,128,"EHMR HABIBI, C.A.",58416411,-642.00,29/04/2026,2026-04-29,-13,-642.00,0.00,"Factura 128 - EHMR HABIBI, C.A.",skipped,saldo_no_positivo,-642.00
4,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,118,"EHMR HABIBI, C.A.",58416411,-242.01,25/04/2026,2026-04-25,-9,-1150.00,907.99,"Factura 118 - EHMR HABIBI, C.A.",skipped,saldo_no_positivo,-242.01
5,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,73,EMPRENDIMIENTO DEYMER DURAN,58412811,-149.00,28/01/2026,2026-01-28,78,-149.00,0.00,Factura 73 - EMPRENDIMIENTO DEYMER DURAN,skipped,saldo_no_positivo,-149.00
6,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,89,EPITACIO MONCADA / MONCADA FERRETERIA LA PRINC...,58416611,-80.84,29/03/2026,2026-03-29,18,-180.00,99.16,Factura 89 - EPITACIO MONCADA / MONCADA FERRET...,skipped,saldo_no_positivo,-80.84
7,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,46,EUTIMIO ALCIDES RIVAS RAMIREZ / RIVAS TODO AGRO,58414611,-1.57,6/11/2025,,161,-520.00,518.43,Factura 46 - EUTIMIO ALCIDES RIVAS RAMIREZ / R...,skipped,saldo_no_positivo,-1.57
8,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,113,FERREAGRO DIAZ PEREZ C.A,58426211,-469.32,16/04/2026,2026-04-16,0,-754.00,284.68,Factura 113 - FERREAGRO DIAZ PEREZ C.A,skipped,saldo_no_positivo,-469.32
9,004 RAMON SAYAGO RUTA FORANEA (EL LLANO),4,116,FERRETERIA ALSEMOC C.A.,58412111,-910.40,16/04/2026,2026-04-16,0,-2000.00,1089.60,Factura 116 - FERRETERIA ALSEMOC C.A.,skipped,saldo_no_positivo,-910.40


""


In [22]:
comparison_path = Path.cwd() / f"comparison_excel_vs_kommo_{run_date_iso}.csv"
kommo_path = Path.cwd() / f"kommo_all_leads_{run_date_iso}.csv"
kommo_cobranza_path = Path.cwd() / f"kommo_cobranza_leads_{run_date_iso}.csv"

comparison_df.to_csv(comparison_path, index=False)
kommo_df.to_csv(kommo_path, index=False)
kommo_cobranza_df.to_csv(kommo_cobranza_path, index=False)

pd.DataFrame([
    {"file": "comparison", "path": str(comparison_path)},
    {"file": "kommo_all_leads", "path": str(kommo_path)},
    {"file": "kommo_cobranza_leads", "path": str(kommo_cobranza_path)},
])


,file,path
0,comparison,c:\Users\luis_\OneDrive\Desktop\KommoAgroriego...
1,kommo_all_leads,c:\Users\luis_\OneDrive\Desktop\KommoAgroriego...
2,kommo_cobranza_leads,c:\Users\luis_\OneDrive\Desktop\KommoAgroriego...


In [ ]:
local_group_df = (
    expected_df
    .groupby(["expected_status_id", "expected_status_name"], dropna=False)
    .size()
    .reset_index(name="cantidad_local")
    .sort_values(["expected_status_name", "expected_status_id"])
    .reset_index(drop=True)
)
display(local_group_df)

,expected_status_id,expected_status_name,cantidad_local
0,104161951,atrasado_10d,1
1,104161955,atrasado_15d,25
2,103388979,atrasado_5d,15
3,103388975,fecha_limite,25
4,103388967,leads_importados,230
5,103388971,recordatorio_enviado,7


In [24]:

kommo_group_df = (
    kommo_cobranza_df
    .groupby(["status_id", "status_name"], dropna=False)
    .size()
    .reset_index(name="cantidad_kommo")
    .sort_values(["status_name", "status_id"])
    .reset_index(drop=True)
)
display(kommo_group_df)

,status_id,status_name,cantidad_kommo
0,104161951,atrasado_10d,1
1,104161955,atrasado_15d,25
2,103388979,atrasado_5d,15
3,103388975,fecha_limite,25
4,103388967,leads_importados,230
5,103388971,recordatorio_enviado,7


In [27]:
base_date = pd.Timestamp(run_date_iso).normalize()

stage_names = {
    103388967: "leads_importados",
    103388971: "recordatorio_enviado",
    103388975: "fecha_limite",
    103388979: "atrasado_5d",
    104161951: "atrasado_10d",
    104161955: "atrasado_15d",
    103429439: "pagado",
    103610175: "abono",
    103611443: "no_pagado",
    103610171: "deadline_abono",
    103429435: "revisar_pago",
    103429431: "revision_urgente",
}

status_ids = {
    "leads_importados": 103388967,
    "recordatorio_enviado": 103388971,
    "fecha_limite": 103388975,
    "atrasado_5d": 103388979,
    "atrasado_10d": 104161951,
    "atrasado_15d": 104161955,
    "pagado": 103429439,
    "abono": 103610175,
    "no_pagado": 103611443,
    "deadline_abono": 103610171,
    "revisar_pago": 103429435,
    "revision_urgente": 103429431,
}

work_df = kommo_cobranza_df.copy()

work_df["nombre"] = work_df["razon_social"].where(
    work_df["razon_social"].astype(str).str.strip().ne(""),
    work_df["name"],
)

work_df["fecha_venc"] = pd.to_datetime(work_df["fecha_venc_iso"], errors="coerce")
work_df["saldo_pendiente"] = pd.to_numeric(work_df["saldo_pendiente"], errors="coerce").fillna(0.0)

for col in ["aviso_3d", "aviso_2d", "aviso_1d"]:
    work_df[col] = work_df[col].fillna("").astype(str)

work_df["sim_status_id"] = pd.to_numeric(work_df["status_id"], errors="coerce").fillna(0).astype(int)
work_df["sim_status_name"] = work_df["status_name"].fillna("").astype(str)

future_dates = [
    (base_date + pd.Timedelta(days=1)).normalize(),
    (base_date + pd.Timedelta(days=2)).normalize(),
    (base_date + pd.Timedelta(days=3)).normalize(),
]

for run_date in future_dates:
    out_col = f"estado_{run_date.strftime('%Y-%m-%d')}"
    work_df[out_col] = work_df["sim_status_name"]

    for idx, row in work_df.iterrows():
        if row["saldo_pendiente"] <= 0:
            continue

        if pd.isna(row["fecha_venc"]):
            continue

        current_status_id = int(row["sim_status_id"])
        due_date = row["fecha_venc"].date()
        d = (due_date - run_date.date()).days

        first_sent = bool(str(row["aviso_3d"]).strip())
        due_sent = bool(str(row["aviso_2d"]).strip())
        final_sent = bool(str(row["aviso_1d"]).strip())

        is_in_base = current_status_id == status_ids["leads_importados"]
        is_in_review = current_status_id == status_ids["revisar_pago"]
        is_in_abono = current_status_id == status_ids["abono"]
        is_in_no_pagado = current_status_id == status_ids["no_pagado"]

        stage_target = current_status_id
        mark_field = ""

        if is_in_review:
            stage_target = current_status_id

        elif is_in_abono:
            if d <= 0 and d > -5:
                stage_target = status_ids["deadline_abono"]
                if not due_sent:
                    mark_field = "aviso_2d"
            elif d <= -5:
                if d <= -15:
                    stage_target = status_ids["atrasado_15d"]
                elif d <= -10:
                    stage_target = status_ids["atrasado_10d"]
                else:
                    stage_target = status_ids["atrasado_5d"]
                    if not final_sent:
                        mark_field = "aviso_1d"

        elif is_in_no_pagado:
            if d <= 0 and d > -5:
                stage_target = status_ids["fecha_limite"]
                if not due_sent:
                    mark_field = "aviso_2d"
            elif d <= -5:
                if d <= -15:
                    stage_target = status_ids["atrasado_15d"]
                elif d <= -10:
                    stage_target = status_ids["atrasado_10d"]
                else:
                    stage_target = status_ids["atrasado_5d"]
                    if not final_sent:
                        mark_field = "aviso_1d"

        elif d == 5 and ((not first_sent) or is_in_base):
            stage_target = status_ids["recordatorio_enviado"]
            if not first_sent:
                mark_field = "aviso_3d"

        elif d <= 0 and d > -5 and ((not due_sent) or is_in_base):
            stage_target = status_ids["fecha_limite"]
            if d == 0 and not due_sent:
                mark_field = "aviso_2d"

        elif d <= -5 and ((not final_sent) or is_in_base):
            if d <= -15:
                stage_target = status_ids["atrasado_15d"]
            elif d <= -10:
                stage_target = status_ids["atrasado_10d"]
            else:
                stage_target = status_ids["atrasado_5d"]
                if not final_sent:
                    mark_field = "aviso_1d"

        next_status_name = stage_names.get(stage_target, str(stage_target))

        work_df.at[idx, out_col] = next_status_name
        work_df.at[idx, "sim_status_id"] = stage_target
        work_df.at[idx, "sim_status_name"] = next_status_name

        if mark_field:
            work_df.at[idx, mark_field] = run_date.strftime("%Y-%m-%dT09:00:00")

resultado_df = work_df[
    [
        "nombre",
        "fecha_venc",
        "status_name",
        f"estado_{future_dates[0].strftime('%Y-%m-%d')}",
        f"estado_{future_dates[1].strftime('%Y-%m-%d')}",
        f"estado_{future_dates[2].strftime('%Y-%m-%d')}",
    ]
].copy()

resultado_df = resultado_df.rename(
    columns={
        "status_name": "estado_actual",
        f"estado_{future_dates[0].strftime('%Y-%m-%d')}": "estado_mañana",
        f"estado_{future_dates[1].strftime('%Y-%m-%d')}": "estado_dia_2",
        f"estado_{future_dates[2].strftime('%Y-%m-%d')}": "estado_dia_3",
    }
)

resultado_df["fecha_venc"] = resultado_df["fecha_venc"].dt.strftime("%Y-%m-%d").fillna("")
resultado_df = resultado_df.sort_values(["fecha_venc", "nombre"]).reset_index(drop=True)

display(resultado_df)


,nombre,fecha_venc,estado_actual,estado_mañana,estado_dia_2,estado_dia_3
0,FERRE MATERIALES EL AGRICULTOR C.A. FEMACA C.A,2024-06-06,atrasado_15d,atrasado_15d,atrasado_15d,atrasado_15d
1,ALAIN JESUS MONTAÑEZ CONTRERAS,2024-11-06,atrasado_15d,atrasado_15d,atrasado_15d,atrasado_15d
2,"SUPERMERCADO LOS RUICES EL MORALITO, C.A.",2024-11-15,atrasado_15d,atrasado_15d,atrasado_15d,atrasado_15d
3,JACINTO ANTONIO LARA VERGARA / VARIEDADES JACI...,2025-03-08,leads_importados,leads_importados,leads_importados,leads_importados
4,BERNAVE SALAS MARQUINA / TRUPERMAX,2025-05-03,atrasado_15d,atrasado_15d,atrasado_15d,atrasado_15d
...,...,...,...,...,...,...
298,NORELYS ANTONIA RODRIGUEZ AZUAJE / INVERSIONES...,2026-05-15,leads_importados,leads_importados,leads_importados,leads_importados
299,EMPRENDIMIENTO PEDRO GUEDEZ,2026-05-16,leads_importados,leads_importados,leads_importados,leads_importados
300,EPITACIO MONCADA / MONCADA FERRETERIA LA PRINC...,2026-05-16,leads_importados,leads_importados,leads_importados,leads_importados
301,FERRE-AGROINSUMOS REYCON C.A,2026-05-16,leads_importados,leads_importados,leads_importados,leads_importados


In [28]:
resultado_df[resultado_df["estado_actual"]!=resultado_df["estado_mañana"]]

,nombre,fecha_venc,estado_actual,estado_mañana,estado_dia_2,estado_dia_3
51,TORNIFERREAGRO KEMEL C.A,2026-04-04,atrasado_10d,atrasado_15d,atrasado_15d,atrasado_15d
106,ELIZABETH GUALDRON DE MUÑOZ / FERRETERIA EL LEON,2026-04-19,leads_importados,fecha_limite,fecha_limite,fecha_limite
107,EMPRENDIMIENTO DEYMER DURAN,2026-04-19,leads_importados,fecha_limite,fecha_limite,fecha_limite
108,"FERREAGRO DON ARMANDO, C.A",2026-04-19,leads_importados,fecha_limite,fecha_limite,fecha_limite
109,YIRIS MADRID GONZALEZ / MADRID FERRETODO,2026-04-19,leads_importados,fecha_limite,fecha_limite,fecha_limite
128,CLIENTE EVENTUAL BARINAS,2026-04-24,leads_importados,recordatorio_enviado,recordatorio_enviado,recordatorio_enviado
129,"TUBOSISTEMAS BARINAS, C.A.",2026-04-24,leads_importados,recordatorio_enviado,recordatorio_enviado,recordatorio_enviado
130,"TUBOSISTEMAS BARINAS, C.A.",2026-04-24,leads_importados,recordatorio_enviado,recordatorio_enviado,recordatorio_enviado
131,"TUBOSISTEMAS BARINAS, C.A.",2026-04-24,leads_importados,recordatorio_enviado,recordatorio_enviado,recordatorio_enviado
